# Wardrobe-AI virtual try-on — free GPU on Colab

Runs the try-on model on Colab's free GPU and prints a public address
your Wardrobe-AI backend can call.

**Use this instead of a Hugging Face ZeroGPU Space when your Hugging
Face account is less than 30 days old**, which is ZeroGPU's requirement
for hosting. Same model, same API, no waiting and no cost.

### Before you start

**Runtime → Change runtime type → Hardware accelerator: T4 GPU.** Cell 1
checks this and stops if it is missing — on a CPU runtime a single
try-on takes many minutes instead of seconds.

### How to use it

1. Run all four cells, top to bottom. The first run installs the model
   and downloads about 2 GB of weights, so give it 5–10 minutes.
2. The last cell prints a line like
   `Running on public URL: https://abc123def456.gradio.live`
3. On the machine running the Wardrobe-AI backend, put that whole
   `https://...gradio.live` address in `.env` as the **fallback**:
   ```
   python -m backend.set_secret TRYON_FALLBACK_URL
   ```
   With `TRYON_PROVIDERS=fashn_space,self_hosted` the backend uses the
   FASHN Space first and this notebook only when the Space is out of
   free quota, busy or unreachable. (To use ONLY this notebook, set
   `TRYON_PROVIDERS=self_hosted`.)
4. Restart the backend. The startup banner should say
   `Virtual try-on: configured`.

This is your own free compute, not a way around Hugging Face's quota:
no Hugging Face token is sent to it, and it has its own limits (Colab's
free GPU is not guaranteed, a runtime lasts at most ~12 hours).

### Two things to know

**The address changes every time you run this notebook**, so you must
redo step 3 on each new session. Start the notebook shortly before you
need it — for a demo, right beforehand.

**Keep this tab open.** Colab stops a notebook that has been idle for
around 90 minutes, and the address dies with it. If try-on suddenly
reports that the service cannot be reached, this notebook has stopped;
re-run the last cell and set the new address.

Treat the share link as a credential while it lives: anyone who has it
can use your GPU session.


## 1. Check the GPU

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode != 0:
    raise SystemExit(
        "No GPU in this runtime.\n\n"
        "Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, "
        "then run this cell again.\n"
        "On a CPU runtime one try-on takes many minutes, which is not "
        "usable."
    )

print(result.stdout.split("\n")[8] if len(result.stdout.split("\n")) > 8
      else result.stdout)
print("GPU found - carry on.")


## 2. Install the model

FASHN VTON v1.5, Apache-2.0 — chosen over IDM-VTON, CatVTON and
OOTDiffusion because those are CC BY-NC-SA and forbid commercial use.


In [ ]:
!pip install -q "git+https://github.com/fashn-AI/fashn-vton-1.5.git" gradio huggingface_hub

print("installed")


## 3. Download the weights

About 2 GB, cached for the rest of this session.


In [ ]:
from huggingface_hub import snapshot_download

WEIGHTS_DIR = "./weights"

snapshot_download(repo_id="fashn-ai/fashn-vton-1.5", local_dir=WEIGHTS_DIR)

from fashn_vton import TryOnPipeline

pipeline = TryOnPipeline(weights_dir=WEIGHTS_DIR)

print("model loaded")


## 4. Start the service

Prints the public address. Copy the whole `https://...gradio.live` line.

The endpoint is named `try_on` and takes three inputs in this order:
person image, garment image, category (`tops`, `bottoms` or
`one-pieces`). That contract is what `backend/virtual_tryon.py` expects
— changing it breaks the backend.


In [ ]:
import gradio as gr

CATEGORIES = ["tops", "bottoms", "one-pieces"]


def try_on(person_image, garment_image, category):
    """One garment onto one person. The backend chains calls for a
    full outfit - bottom first, then top applied to that result."""
    if person_image is None:
        raise gr.Error("No photo of the person was supplied.")

    if garment_image is None:
        raise gr.Error("No garment image was supplied.")

    if category not in CATEGORIES:
        raise gr.Error(
            f"Category must be one of {', '.join(CATEGORIES)} - got "
            f"'{category}'."
        )

    result = pipeline(
        person_image=person_image.convert("RGB"),
        garment_image=garment_image.convert("RGB"),
        category=category,
    )

    return result.images[0]


with gr.Blocks(title="Wardrobe-AI Try-On") as demo:

    gr.Markdown("## Wardrobe-AI virtual try-on (Colab GPU)")

    with gr.Row():
        person_input = gr.Image(label="Person", type="pil")
        garment_input = gr.Image(label="Garment", type="pil")
        output_image = gr.Image(label="Result", type="pil")

    category_input = gr.Dropdown(CATEGORIES, value="tops", label="Category")

    run_button = gr.Button("Try on", variant="primary")

    # api_name is the contract. backend/virtual_tryon.py asks for
    # "/try_on" by name.
    run_button.click(
        fn=try_on,
        inputs=[person_input, garment_input, category_input],
        outputs=output_image,
        api_name="try_on",
    )

# share=True is what makes this reachable from your laptop's backend.
demo.queue().launch(share=True)
